![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG with predefined Milvus index to create a pattern about IBM

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate the usage of IBM AutoAI RAG with predefined vector store collection. Although this example uses Milvus, Elasticsearch and Chroma databases can be used similarly. Note that the AutoAI RAG experiment conducted in this notebook uses data scraped from the `ibm-watsonx-ai` SDK documentation.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The learning goals of this notebook are:

- Create an AutoAI RAG job that will find the best RAG pattern based on collection created from `ibm-watsonx-ai` SDK documentation.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Index creation](#Index-creation)
3. [RAG Optimizer definition](#RAG-Optimizer-definition)
4. [RAG Experiment run](#RAG-Experiment-run)
5. [Comparison and testing of RAG Patterns](#Comparison-and-testing-of-RAG-Patterns)
6. [Historical runs](#Historical-runs)
7. [Cleanup](#Cleanup)
8. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import the required modules and dependencies

In [1]:
%pip install -U "ibm-watsonx-ai[rag]>=1.4.6" | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
import os

try:
    USERNAME = os.environ["USERNAME"]
except KeyError:
    USERNAME = input("Please enter your username (hit enter): ")

try:
    URL = os.environ["URL"]
except KeyError:
    URL = input("Please enter the platform url (hit enter): ")

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=USERNAME,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=URL,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [4]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=USERNAME,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=URL,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, you need to create a space for your work. If you do not have a space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Go to **Manage** tab
- Copy `Space GUID` into your env file or else enter it in the window which will show up after running below cell

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below

In [6]:
import os

try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

Set your space as default.

In [7]:
client.set.default_space(space_id)

'SUCCESS'

<a id="Index-creation"></a>
## Index creation

### Defining a connection to knowledge base

Provide id of connection to your knowledge database or create a new one. You can add connection on watsonx platform or type your credentials after running the code below.

In [8]:
vector_store_connection_id = (
    input(
        "Provide connection asset ID in your space. Skip this, if you wish to type credentials by hand and hit enter: "
    )
    or None
)

if vector_store_connection_id is None:
    try:
        username = os.environ["MILVUS_USER"]
    except KeyError:
        username = input("Please enter your Milvus user name and hit enter: ")
    try:
        password = os.environ["MILVUS_PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your Milvus password and hit enter: ")
    try:
        host = os.environ["MILVUS_HOST"]
    except KeyError:
        host = input("Please enter your Milvus hostname and hit enter: ")
    try:
        port = os.environ["MILVUS_PORT"]
    except KeyError:
        port = input("Please enter your Milvus port number and hit enter: ")
    try:
        ssl = os.environ["MILVUS_SSL"]
    except:
        ssl = bool(
            input(
                "Please enter ('y'/anything) if your Milvus instance has SSL enabled. Skip if it is not: "
            )
        )

    # Create connection
    milvus_data_source_type_id = client.connections.get_datasource_type_uid_by_name(
        "milvus"
    )
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Milvus Connection - sample notebook",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: milvus_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": host,
                "port": port,
                "username": username,
                "password": password,
                "ssl": ssl,
            },
        }
    )

    vector_store_connection_id = client.connections.get_id(details)

Creating connections...
SUCCESS


Download example data. You can also assign your own text to `document content`.

In [9]:
import requests

url = "https://ibm.github.io/watsonx-ai-python-sdk/v1.3.42/base.html"

response = requests.get(url)
response.raise_for_status()

document_content = response.text

Chunk and upload your document to the vector store.

In [10]:
from ibm_watsonx_ai.foundation_models.embeddings import Embeddings
from ibm_watsonx_ai.foundation_models.extensions.rag.chunker import LangChainChunker
from ibm_watsonx_ai.foundation_models.extensions.rag.vector_stores import (
    MilvusVectorStore,
)
from langchain_core.documents import Document

# Defining vector store from the connection id
embedding = Embeddings(model_id="ibm/slate-125m-english-rtrvr", api_client=client)
vector_store = MilvusVectorStore(
    api_client=client,
    connection_id=vector_store_connection_id,
    collection_name="collection_notebook_sample",
    embedding_function=embedding,
    drop_old=True,
)

# Chunking document into smaller segments
document = Document(
    page_content=document_content, metadata={"document_id": "base.html"}
)
text_splitter = LangChainChunker(method="recursive", chunk_size=256, chunk_overlap=32)
chunks = text_splitter.split_documents([document])

# Uploading document to vector store
ids = vector_store.add_documents(chunks, batch_size=300)

print(ids[:5])

['8b427f0c5f8937f53cb044e8f33eef298554344625d456dea268c6ad2474f01f', '3894eb5d7dea204d68490e08a17d87b151c8a517e14d2585574d75b1fac0d365', '038e7d15480b0e0169e4642efa5a4d4abd5b976d42f459d7732cd3037e604991', '6c904049a07ffe19a594217e0acfb6bad47716e8a52c2afd5d5a30c7edd883a5', 'd745774f5ac8e2e12a071716cb284158b57291e65cdd42ead8ee6b93950f8d00']


<a id="RAG-Optimizer-definition"></a>
## RAG Optimizer definition

### Defining a connection to vector store

Define a reference to knowledge base.

In [11]:
from ibm_watsonx_ai.helpers import DataConnection
from ibm_watsonx_ai.utils.autoai.enums import KnowledgeBaseFieldRole
from ibm_watsonx_ai.utils.autoai.knowledge_base import VectorStoreKnowledgeBase

connection = DataConnection(connection_asset_id=vector_store_connection_id)
connection.set_client(client)

vector_store_knowledge_base_references = [
    VectorStoreKnowledgeBase(
        name="Embedded base.html file",
        description="This knowledge base contains samples from watsonx.ai sdk documentation.",
        connection=connection,
        settings={
            "index_name": "collection_notebook_sample",
            "fields_mapping": [
                {
                    "role": KnowledgeBaseFieldRole.DENSE_VECTOR_EMBEDDINGS,
                    "field_name": "vector",
                },
                {
                    "role": KnowledgeBaseFieldRole.DOCUMENT_NAME,
                    "field_name": "document_id",
                },
                {
                    "role": KnowledgeBaseFieldRole.TEXT,
                    "field_name": "text",
                },
                {
                    "role": KnowledgeBaseFieldRole.CHUNK_SEQUENCE_NUMBER,
                    "field_name": "sequence_number",
                },
            ],
            "embeddings": {"model_id": "ibm/slate-125m-english-rtrvr"},
        },
    )
]

### Defining a connection to test data

Upload a `json` file that will be used for benchmarking to COS and then define a connection to this file. 
Define benchmarking question about your knowledge base. Replace the questions below.

In [12]:
benchmarking_data_IBM_page_content = [
    {
        "question": "How can you set or refresh user request headers using the APIClient class?",
        "correct_answer": "client.set_headers({'Authorization': 'Bearer <token>'})",
        "correct_answer_document_ids": ["base.html"],
    },
    {
        "question": "How to initialise Credentials object with api_key",
        "correct_answer": "credentials = Credentials(url = 'https://us-south.ml.cloud.ibm.com', api_key = '***********')",
        "correct_answer_document_ids": ["base.html"],
    },
]

Upload testing data to the bucket as a `json` file.

In [13]:
import json

test_filename = "benchmarking_data_predefined_vector_store_sample.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data_IBM_page_content, json_file, indent=4)

test_asset_details = client.data_assets.create(
    name=test_filename, file_path=test_filename
)

test_asset_id = client.data_assets.get_id(test_asset_details)
test_asset_id

Creating data asset...
SUCCESS


'01a066bd-38ff-765b-983c-225df19c0a10'

Define connection information to testing data.

In [14]:
test_data_references = [DataConnection(data_asset_id=test_asset_id)]

### RAG Optimizer configuration

Provide the input information for AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [15]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.foundation_models.schema import (
    AutoAIRAGGenerationConfig,
    AutoAIRAGModelConfig,
)

experiment = AutoAI(
    credentials=credentials,
    space_id=space_id,
)

foundation_model = AutoAIRAGModelConfig(
    model_id="ibm/granite-4-h-small",
)

generation_config = AutoAIRAGGenerationConfig(
    foundation_models=[foundation_model],
)

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG - sample notebook - knowledge base",
    description="Experiment run in sample notebook",
    generation=generation_config,
    max_number_of_rag_patterns=3,
    optimization_metrics=[AutoAI.RAGMetrics.ANSWER_CORRECTNESS],
)

Configuration parameters can be retrieved via `get_params()`.

In [16]:
rag_optimizer.get_params()

{'name': 'AutoAI RAG - sample notebook - knowledge base',
 'description': 'Experiment run in sample notebook',
 'max_number_of_rag_patterns': 3,
 'optimization_metrics': ['answer_correctness']}

<a id="RAG-Experiment-run"></a>
## RAG Experiment run

Call the `run()` method to trigger the AutoAI RAG experiment. You can either use interactive mode (synchronous job) or background mode (asynchronous job) by specifying `background_mode=True`.

In [17]:
run_details = rag_optimizer.run(
    knowledge_base_references=vector_store_knowledge_base_references,
    test_data_references=test_data_references,
    background_mode=False,
)



##############################################

Running 'fe6288f7-20b5-49e6-8bf4-4779db20ccb8'

##############################################


pending...........
running.......
completed
Training of 'fe6288f7-20b5-49e6-8bf4-4779db20ccb8' finished successfully.


You can use the `get_run_status()` method to monitor AutoAI RAG jobs in background mode.

In [18]:
rag_optimizer.get_run_status()

'completed'

<a id="Comparison-and-testing-of-RAG-Patterns"></a>
## Comparison and testing of RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. You can use the DataFrame to compare all discovered patterns and select the one you like for further testing.

In [19]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,generation.model_id,agent.type
Pattern_Name,,,
Pattern1,0.7083,ibm/granite-4-h-small,sequential
Pattern2,0.5417,ibm/granite-4-h-small,sequential
Pattern3,0.5417,ibm/granite-4-h-small,sequential


Additionally, you can pass the `scoring` parameter to the summary method, to filter RAG patterns starting with the best.

```python
summary = rag_optimizer.summary(scoring="answer_correctness")
```

In [20]:
rag_optimizer.get_run_details()

{'entity': {'hardware_spec': {'id': 'a6c4923b-b8e4-444c-9f43-8a7ec3020110',
   'name': 'L'},
  'knowledge_base_references': [{'description': 'This knowledge base contains samples from watsonx.ai sdk documentation.',
    'name': 'Embedded base.html file',
    'reference': {'connection': {'id': '01a066b7-f6b7-728d-87cb-983bef599012'},
     'location': {},
     'type': 'connection_asset'},
    'settings': {'embeddings': {'model_id': 'ibm/slate-125m-english-rtrvr'},
     'fields_mapping': [{'field_name': 'vector',
       'role': 'dense_vector_embeddings'},
      {'field_name': 'document_id', 'role': 'document_name'},
      {'field_name': 'text', 'role': 'text'},
      {'field_name': 'sequence_number', 'role': 'chunk_sequence_number'}],
     'index_name': 'collection_notebook_sample'},
    'type': 'vector_store'}],
  'parameters': {'constraints': {'max_number_of_rag_patterns': 3},
   'optimization': {'metrics': ['answer_correctness']},
   'output_logs': True},
  'results': [{'context': {'it

### Get selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [21]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern()

Best pattern is: Pattern1


The pattern details can be retrieved by calling the `get_pattern_details` method:

```python
rag_optimizer.get_pattern_details(pattern_name='Pattern2')
```

Query the RAGPattern locally, to test it.

In [22]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)
inference_service_function = best_pattern.inference_service(
    runtime_context, url=client.credentials.url
)[0]

In [23]:
question = "How to add Task Credentials?"

context = RuntimeContext(
    api_client=client,
    request_payload_json={"messages": [{"role": "user", "content": question}]},
)

inference_service_function(context)

{'body': {'choices': [{'index': 0,
    'message': {'role': 'system',
     'content': 'To add Task Credentials using the IBM watsonx.ai Python SDK, you can follow these steps:\n\n1. Import the necessary classes:\n```python\nfrom ibm_watsonx_ai import Credentials, APIClient\n```\n\n2. Create an instance of the `Credentials` class by providing the required parameters:\n```python\ncredentials = Credentials(\n    url="<url>",\n    api_key=IAM_API_KEY\n)\n```\nReplace `<url>` with the actual URL of your IBM Cloud instance, and `IAM_API_KEY` with your IBM Cloud API key.\n\nAlternatively, you can create the `Credentials` instance using a dictionary:\n```python\ncredentials = Credentials.from_dict({\n    \'url\': "<url>",\n    \'apikey\': IAM_API_KEY\n})\n```\n\n3. Create an instance of the `APIClient` class by passing the `credentials` object:\n```python\nclient = APIClient(credentials)\n```\n\n4. (Optional) If you want to set a specific space ID for the client, you can do so by passing the `s

### Deploy RAGPattern

Deployment is done by storing the defined RAG function and then by creating a deployed asset.

In [24]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentation",
    space_id=space_id,
    deploy_params={"tags": ["wx-autoai-rag"]},
)



######################################################################################

Synchronous deployment creation for id: '01a066c4-7928-72db-8de9-2185bbcfb21a' started

######################################################################################


initializing
Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
......
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='01a066c4-9bc7-754f-92ce-9d7706985c5e'
-----------------------------------------------------------------------------------------------




### Test the deployed function

RAG service is now deployed in our space. To test our solution we can run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [25]:
deployment_id = client.deployments.get_id(deployment_details)

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)

In [26]:
print(score_response["choices"][0]["message"]["content"])

To add Task Credentials for IBM watsonx.ai using the Python SDK, you can follow these steps:

1. Import the necessary modules:
   ```python
   from ibm_watsonx_ai import Credentials
   ```

2. Create an instance of the `Credentials` class by providing the required parameters:
   - `url`: The URL of the IBM watsonx.ai service endpoint.
   - `api_key`: Your IBM Cloud API key.

   Example:
   ```python
   credentials = Credentials(
       url="https://us-south.ml.cloud.ibm.com",
       api_key="YOUR_API_KEY"
   )
   ```

   Alternatively, you can create the `Credentials` object using a dictionary:
   ```python
   credentials = Credentials.from_dict({
       'url': "https://us-south.ml.cloud.ibm.com",
       'apikey': "YOUR_API_KEY"
   })
   ```

3. Use the `credentials` object when creating an instance of the `APIClient`:
   ```python
   from ibm_watsonx_ai import APIClient

   client = APIClient(credentials, space_id="YOUR_SPACE_ID")
   ```

   Replace `"YOUR_SPACE_ID"` with the actual s

<a id="Historical-runs"></a>
## Historical runs

In this section you learn to work with historical RAG Optimizer jobs (runs).

To list historical runs use the `list()` method and provide the `'rag_optimizer'` filter.

In [27]:
experiment.runs(filter="rag_optimizer").list()

,timestamp,run_id,state,auto_pipeline_optimizer name
0,2026-09-03T10:13:41.939Z,fe6288f7-20b5-49e6-8bf4-4779db20ccb8,completed,AutoAI RAG - sample notebook - knowledge base


In [28]:
run_id = run_details["metadata"]["id"]
run_id

'fe6288f7-20b5-49e6-8bf4-4779db20ccb8'

### Get executed optimizer's configuration parameters

In [29]:
experiment.runs.get_rag_params(run_id=run_id)

{'name': 'AutoAI RAG - sample notebook - knowledge base',
 'description': 'Experiment run in sample notebook',
 'max_number_of_rag_patterns': 3,
 'optimization_metrics': ['answer_correctness']}

### Get historical rag_optimizer instance and training details

In [30]:
historical_opt = experiment.runs.get_rag_optimizer(run_id)

### List trained patterns for selected optimizer

In [31]:
historical_opt.summary()

,mean_answer_correctness,generation.model_id,agent.type
Pattern_Name,,,
Pattern1,0.7083,ibm/granite-4-h-small,sequential
Pattern2,0.5417,ibm/granite-4-h-small,sequential
Pattern3,0.5417,ibm/granite-4-h-small,sequential


<a id="Cleanup"></a>
## Cleanup

To delete the current experiment, use the `cancel_run` method.

**Warning:** Be careful: once you delete an experiment, you will no longer be able to refer to it.

In [32]:
rag_optimizer.cancel_run(hard_delete=True)

'SUCCESS'

To delete the deployment, use the `delete` method. 

**Warning:** Keeping the deployment active may lead to unnecessary consumption of Compute Unit Hours (CUHs).

In [33]:
client.deployments.delete(deployment_id)

'SUCCESS'

To delete obsolete collections, us the `clear` method of `MilvusVectorStore`

In [34]:
vector_store.clear()

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI RAG experiments. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Paweł Kocur**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.